# Emerging Technologies - Assessment Problems

This notebook explores the difference between classical and quantum algorithms through five problems centred on the [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm). The Deutsch-Jozsa algorithm is historically significant as one of the first demonstrations that a quantum computer can solve a well-defined problem exponentially faster than any classical deterministic algorithm.

The problems progress from classical Python to full Qiskit quantum circuits:

1. Generating random Boolean functions
2. Classical algorithm to determine function type
3. Quantum oracles for Deutsch's single-input problem
4. Deutsch's algorithm implemented in Qiskit
5. Generalisation to the full Deutsch-Jozsa algorithm for four-bit inputs

In [2]:
# Standard library: random selections and combinatorics.
import random
import itertools as it

# Quantum computing framework.
# See: https://docs.quantum.ibm.com/
import qiskit

# Quantum circuit simulator.
# See: https://qiskit.github.io/qiskit-aer/
import qiskit_aer as aer

# Quantum state and visualisation utilities.
import qiskit.quantum_info as info
import qiskit.visualization as viz

---

## Problem 1: Generating Random Boolean Functions

### Background

A **Boolean function** with $n$ inputs maps binary strings to a single bit:

$$f : \{0, 1\}^n \to \{0, 1\}$$

The [Deutsch-Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm) is guaranteed that the function is one of two types:

- **Constant**: $f(x) = 0$ for all inputs, or $f(x) = 1$ for all inputs. There are exactly **2** constant functions.
- **Balanced**: $f(x) = 1$ for exactly **half** of all possible inputs. For $n = 4$ inputs ($2^4 = 16$ combinations), exactly 8 outputs must be 1. The number of balanced functions is $\binom{16}{8} = 12{,}870$.

As described in the [IBM Quantum course on classical information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/single-systems/classical-information#deterministic-operations), Boolean functions form the foundation of both classical and quantum computation.

### Implementation Strategy

The function uses a **lookup table** approach: all $2^4 = 16$ input combinations are precomputed, outputs are assigned (constant or balanced), and the result is returned as a Python **closure**. A closure is an inner function that retains access to variables from its enclosing scope - see [Real Python: Python Closures](https://realpython.com/python-closure/). This makes the function appear opaque to the caller: the type cannot be determined by inspecting the function object, only by evaluating it on inputs.

In [4]:
def random_constant_balanced():
    """Return a randomly chosen constant or balanced function.

    The returned function accepts exactly four Boolean arguments
    and returns 0 or 1.

    Constant: all 16 outputs are 0, or all are 1.
    Balanced: exactly 8 of the 16 outputs are 1.

    Returns:
        f (callable): A function f(a, b, c, d) -> int (0 or 1).
    """
    # Generate every 4-bit input combination: (0,0,0,0), (0,0,0,1), ..., (1,1,1,1).
    # itertools.product with repeat=4 gives 2^4 = 16 tuples.
    all_inputs = list(it.product((0, 1), repeat=4))

    # Randomly select function type with equal probability.
    func_type = random.choice(['constant', 'balanced'])

    if func_type == 'constant':
        # Constant: choose either always-0 or always-1.
        value = random.choice([0, 1])
        lookup = {inp: value for inp in all_inputs}
    else:
        # Balanced: exactly 8 outputs are 1, 8 are 0.
        # Start with an equal split, then shuffle to randomise placement.
        outputs = [0] * 8 + [1] * 8
        random.shuffle(outputs)
        lookup = dict(zip(all_inputs, outputs))

    def f(a, b, c, d):
        """Return f(a, b, c, d) from the precomputed lookup table."""
        return lookup[(a, b, c, d)]

    # Attach the type as metadata 
    f._type = func_type
    return f

### Demonstration

We generate a function and print its full truth table. The `f._type` attribute is attached for testing purposes only - in the real Deutsch-Jozsa scenario the caller has no way to inspect this.

In [5]:
# Generate a random constant-or-balanced function.
f_demo = random_constant_balanced()

# Print the full truth table (16 rows for n=4).
print(f"{'a':>3} {'b':>3} {'c':>3} {'d':>3}  |  f(a,b,c,d)")
print("-" * 32)
for inp in it.product((0, 1), repeat=4):
    print(f"{inp[0]:>3} {inp[1]:>3} {inp[2]:>3} {inp[3]:>3}  |  {f_demo(*inp)}")

  a   b   c   d  |  f(a,b,c,d)
--------------------------------
  0   0   0   0  |  0
  0   0   0   1  |  0
  0   0   1   0  |  1
  0   0   1   1  |  1
  0   1   0   0  |  0
  0   1   0   1  |  1
  0   1   1   0  |  0
  0   1   1   1  |  0
  1   0   0   0  |  1
  1   0   0   1  |  0
  1   0   1   0  |  1
  1   0   1   1  |  0
  1   1   0   0  |  0
  1   1   0   1  |  1
  1   1   1   0  |  1
  1   1   1   1  |  1


In [6]:
# Verify the output distribution.
all_outputs = [f_demo(*inp) for inp in it.product((0, 1), repeat=4)]
ones = sum(all_outputs)

print(f"Number of 1-outputs : {ones} / 16")
print(f"Number of 0-outputs : {16 - ones} / 16")
print(f"Hidden type         : {f_demo._type}")
print(f"Verified type       : {'constant' if ones in (0, 16) else 'balanced'}")

Number of 1-outputs : 8 / 16
Number of 0-outputs : 8 / 16
Hidden type         : balanced
Verified type       : balanced


In [7]:
# Generate several functions to confirm both types appear.
print("Generating 10 random functions:")
for i in range(10):
    g = random_constant_balanced()
    outputs = [g(*inp) for inp in it.product((0, 1), repeat=4)]
    ones_count = sum(outputs)
    verified = 'constant' if ones_count in (0, 16) else 'balanced'
    print(f"  Function {i+1}: ones={ones_count:>2}, type={verified}")

Generating 10 random functions:
  Function 1: ones=16, type=constant
  Function 2: ones= 0, type=constant
  Function 3: ones= 0, type=constant
  Function 4: ones=16, type=constant
  Function 5: ones= 8, type=balanced
  Function 6: ones= 0, type=constant
  Function 7: ones= 8, type=balanced
  Function 8: ones= 8, type=balanced
  Function 9: ones= 8, type=balanced
  Function 10: ones= 0, type=constant
